# Lumis: evidence-backed triage and a tool-using investigator

Run from the development checkout with `uv sync --extra agent --all-groups`. This notebook uses a scripted Pydantic AI FunctionModel, not a paid provider. It verifies the real tool loop and report protocol, **not diagnosis accuracy**. No consumer repository, network, Docker or credentials are needed. Generated probes are disabled here; see `docs/sandbox.md` for genuine isolated execution qualification. Suggestions are never applied.


In [ ]:
import json
import tempfile
from pathlib import Path

from pydantic import TypeAdapter
from pydantic_ai.messages import ModelResponse, ToolCallPart
from pydantic_ai.models.function import FunctionModel

from lumis_sdk.core import Evidence, Incident
from lumis_sdk.investigation.agent import PydanticInvestigator
from lumis_sdk.runtime import IncidentStore, YamlProject
from lumis_sdk.runtime.scaffold import starter_documents

workspace = Path(tempfile.mkdtemp(prefix="lumis-agent-"))
for name, content in starter_documents().items():
    (workspace / name).write_text(content)
source = workspace / "approved-source"
source.mkdir()
(source / "handler.py").write_text("def ready(healthy):\n    return bool(healthy)\n")
config = json.loads((workspace / "lumis.yaml").read_text())
config["investigator"] = {
    "repositories": [
        {
            "id": "demo-code",
            "root": "approved-source",
            "entity_ids": ["service:demo"],
            "files": ["handler.py"],
        }
    ]
}
(workspace / "lumis.yaml").write_text(json.dumps(config))
project = YamlProject.from_file(workspace / "lumis.yaml")
event = Incident.model_validate_json((workspace / "incident.json").read_text())
facts = TypeAdapter(tuple[Evidence, ...]).validate_json(
    (workspace / "observations.json").read_text()
)
triage_only = await project.handle_incident(event, observations=facts)
assert triage_only.findings[0].status == "match"  # supported but nonterminal
assert triage_only.route == "human"  # no implicit model invocation
print(triage_only.stop_reason)

## A real agent loop with scripted responses

The model first discovers available tools, reads the allowlisted source and registers a candidate. It returns a typed tentative report. Lumis owns observation acquisition and mechanical support. The script sees no hidden fault label, cannot add telemetry, and cannot write the application source.


In [ ]:
candidate = project.config.checks[0].hypothesis
calls = []


def scripted_model(messages, info):
    calls.append(messages)
    requests = [
        {"operation": "catalog"},
        {"operation": "code.read", "target": "demo-code", "path": "handler.py"},
        {"operation": "hypothesis.register", "hypothesis": candidate.model_dump(mode="json")},
    ]
    if len(calls) <= len(requests):
        return ModelResponse(
            parts=[
                ToolCallPart(
                    "inspect",
                    {"request": requests[len(calls) - 1]},
                    tool_call_id=f"inspect-{len(calls)}",
                )
            ]
        )
    return ModelResponse(
        parts=[
            ToolCallPart(
                info.output_tools[0].name,
                {
                    "hypotheses": [candidate.model_dump(mode="json")],
                    "suggestions": [
                        {
                            "hypothesis_id": candidate.id,
                            "description": "Review readiness and health telemetry.",
                            "evidence_ids": ["demo-health"],
                        }
                    ],
                    "unresolved_questions": ["Why did the service become unhealthy?"],
                },
                tool_call_id="report",
            )
        ]
    )


report = await project.handle_incident(
    event, observations=facts, investigator=PydanticInvestigator(FunctionModel(scripted_model))
)
assert report.route == "agent"
assert report.metrics.model_requests == 4
assert report.metrics.tool_attempts == 3
assert report.assessments[0].state == "supported"
assert report.truth_state == "unconfirmed_hypothesis"
assert report.requires_human_review
print(report.model_dump_json(indent=2))

## Store the report, not a claim of automatic recovery

The SQLite record is immutable. A human may separately record a reviewed resolution. It never changes the diagnostic truth state. Paid-provider use is a separate opt-in with an explicit tool-capable model ID; do not substitute it in this notebook without reviewing approved context and costs.


In [ ]:
from lumis_sdk.investigation.contracts import HumanResolution

store = IncidentStore(workspace / "incidents.sqlite")
store.save(report)
resolution = HumanResolution(
    id="manual-review-1",
    incident_id=event.id,
    reviewer="notebook-operator",
    recorded_at=event.ended_at,
    summary="Synthetic demonstration reviewed.",
    applied_change="No change applied.",
    outcome="inconclusive",
)
store.record_resolution(resolution)
assert store.get(event.id) == report
assert store.resolutions(event.id) == (resolution,)
assert (source / "handler.py").read_text().startswith("def ready")
print("Human-review record saved; no recovery or rule promotion performed.")